In [ ]:
import math
import pandas as pd
import rasterio
from pathlib import Path

from rasterio.warp import transform_bounds
from rasterio.transform import from_origin

In [ ]:
DIR_DATA = Path("data")
DIR_ICEYE_ORG = DIR_DATA / "1_org" / "1_ICEYE"
FILEPATH_ICEYE_LIST = DIR_DATA / "2_processed" / "0_metadata" / "iceye-list.csv"

TARGET_CRS = "EPSG:2958"
TARGET_RESOLUTION = 0.5  # metres

In [ ]:
images = pd.read_csv(FILEPATH_ICEYE_LIST)

lefts = []
rights = []
tops = []
bottoms = []

for _, img in images.iterrows():

    if pd.isna(img["filename"]):
        continue

    path = DIR_ICEYE_ORG / img["filename"]
    path = DIR_ICEYE_ORG / (
            path.stem + "_EPSG2958_res05m.tif"
        )

    with rasterio.open(path) as src:

        # Transform bounds into EPSG:2958
        left, bottom, right, top = transform_bounds(
            src.crs,
            TARGET_CRS,
            *src.bounds
        )

        lefts.append(left)
        rights.append(right)
        bottoms.append(bottom)
        tops.append(top)

# --------------------------------------
# Common extent
# --------------------------------------
LEFT = min(lefts)
RIGHT = max(rights)
BOTTOM = min(bottoms)
TOP = max(tops)

print("Common extent")
print(LEFT, RIGHT)
print(BOTTOM, TOP)

# --------------------------------------
# Common raster grid
# --------------------------------------
WIDTH = math.ceil(
    (RIGHT - LEFT) / TARGET_RESOLUTION
)
HEIGHT = math.ceil(
    (TOP - BOTTOM) / TARGET_RESOLUTION
)
COMMON_TRANSFORM = from_origin(
    LEFT,
    TOP,
    TARGET_RESOLUTION,
    TARGET_RESOLUTION
)

print(COMMON_TRANSFORM)
print(WIDTH, HEIGHT)

In [ ]:
from rasterio.warp import reproject
from rasterio.warp import Resampling

In [ ]:
images = pd.read_csv(FILEPATH_ICEYE_LIST)

for _, img in images.iterrows():
    if pd.isna(img["filename"]):
        continue

    image_path = DIR_ICEYE_ORG / img["filename"]
    image_path = DIR_ICEYE_ORG / (
        image_path.stem + "_EPSG2958_res05m.tif"
    )
    with rasterio.open(image_path) as src:
        profile = src.profile.copy()
        profile.update(
            crs=TARGET_CRS,
            transform=COMMON_TRANSFORM,
            width=WIDTH,
            height=HEIGHT
        )
        dst_path = image_path
        with rasterio.open(
            dst_path,
            "w",
            **profile
        ) as dst:
            for band in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, band),
                    destination=rasterio.band(dst, band),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=COMMON_TRANSFORM,
                    dst_crs=TARGET_CRS,
                    resampling=Resampling.bilinear
                )

    print(f"Saved {dst_path.name}")